# Ghaderi2025_Figure3D_PSTH_depth.ipynb

## Manuscript Information
**Title:** Contextual gating of whisker-evoked responses by frontal cortex supports flexible decision making  
**Authors:** Parviz Ghaderi, Sylvain Crochet, Carl Petersen  
**Year:** 2025  
**Code Author:** Parviz Ghaderi  
**Email:** parviz.ghaderi7@gmail.com

## Description
This notebook generates Figure 3D showing PSTH (Peri-Stimulus Time Histogram) analysis across cortical depths. It creates depth-resolved PSTH plots showing neural activity across different cortical layers and brain areas. The analysis includes:

- **Depth-resolved heatmaps** showing activity patterns as a function of cortical depth and time
- **Average PSTH plots** for different trial conditions and cortical layers
- **Multi-area comparison** across five brain areas (A1, wS1, wS2, wM2, ALM)

## Dependencies
- numpy (for numerical computations)
- matplotlib (for plotting)
- scipy (for signal processing)
- pickle (for data loading)
- os (for file operations)

## Input Files

'Neuronal_data_10ms.mat'
## Output
SVG figure showing depth-resolved PSTH analysis

## 1. Import Required Libraries and Configure Settings

In [1]:
# Import required libraries
import os
import pickle
import numpy as np
import matplotlib
matplotlib.use('TkAgg')
import matplotlib.pyplot as plt
from scipy.signal import convolve
from scipy.signal.windows import gaussian
from matplotlib.widgets import RectangleSelector
import matplotlib.colors as mcolors
from scipy.io import loadmat

# Configure matplotlib settings for publication
matplotlib.rcParams['pdf.fonttype'] = 42  # Embed fonts as editable text
matplotlib.rcParams['svg.fonttype'] = 'none'  # Embed fonts as editable text

print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Define Color Schemes and Trial Conditions

In [2]:
# Color codes for different trial conditions
colorcodes = np.array([
    [0,   0,   1  ],  # Blue - Go-tone whisker
    [0, 0.5,   1  ],  # Light blue - Go-tone
    [1,   0,   0  ],  # Red - Nogo-tone whisker
    [1, 0.5,   0  ],  # Orange - Nogo-tone
    [0,   0,   0  ]   # Black - No-tone whisker
])

# Trial condition labels
condition = np.array([
    'Go-tone whisker', 
    'Go-tone', 
    'Nogo-tone whisker', 
    'Nogo-tone', 
    'No-tone whisker'
])

print(f"Defined {len(condition)} trial conditions:")
for i, cond in enumerate(condition):
    print(f"  {i+1}. {cond}")

Defined 5 trial conditions:
  1. Go-tone whisker
  2. Go-tone
  3. Nogo-tone whisker
  4. Nogo-tone
  5. No-tone whisker


## 3. Define Plotting Functions

In [3]:
def plot_heatmap(ax, psth, cluster_indices, time2plot, start_bin, end_bin, depth, layer, bin_size=25):
    """
    Plot depth-resolved heatmap of neural activity.
    
    Parameters:
    -----------
    ax : matplotlib.axes.Axes
        Axis to plot on
    psth : numpy.ndarray
        Peri-stimulus time histogram data
    cluster_indices : numpy.ndarray
        Indices of clusters to include
    time2plot : numpy.ndarray
        Time points for plotting
    start_bin : int
        Starting bin index
    end_bin : int
        Ending bin index
    depth : numpy.ndarray
        Cortical depth values
    layer : numpy.ndarray
        Cortical layer labels
    bin_size : int, optional
        Size of depth bins (default: 25)
    """
    # Define time segment for plotting
    time_segment = time2plot[0:299]
    cdata = psth[cluster_indices, start_bin:end_bin]

    # Define custom colormap with narrow transition around zero
    colors = [
        (0 / 255, 230 / 255, 230 / 255),   # Cyan for very negative values
        (0.0, 0.1, 1),                     # Dark blue near zero (negative side)
        (0.0, 0.0, 0.0),                   # Black at zero
        (1, 0.1, 0.0),                     # Dark red near zero (positive side)
        (1.0, 1.0, 0.0),                   # Yellow for very positive values
    ]

    # Adjust positions to make transition around zero narrower
    positions = [0.0, 0.3, 0.5, 0.7, 1.0]  # Narrow black region around zero

    # Create custom colormap
    custom_cmap = mcolors.LinearSegmentedColormap.from_list(
        "custom_cmap", list(zip(positions, colors)), N=256
    )

    # Bin data by depth
    depth_bins = np.arange(0, np.max(depth) + bin_size, bin_size)
    binned_data = np.zeros((len(depth_bins) - 1, cdata.shape[1]))
    binned_layers = []
    binned_depth = []
    
    for i in range(len(depth_bins) - 1):
        bin_indices = np.where((depth[cluster_indices] >= depth_bins[i]) & 
                              (depth[cluster_indices] < depth_bins[i + 1]))[0]
        if len(bin_indices) > 0:
            binned_data[i, :] = np.mean(cdata[bin_indices, :], axis=0)
            binned_depth = np.hstack((binned_depth, np.mean(depth[cluster_indices][bin_indices])))
            binned_layers = np.hstack((binned_layers, layer[cluster_indices][bin_indices[0]]))
    
    # Create heatmap
    img = ax.imshow(
        binned_data,
        origin="upper",
        aspect="auto",
        extent=[time_segment[0], time_segment[-1], depth_bins[-1], depth_bins[0]],
        vmin=cmin,
        vmax=cmax,
        cmap=custom_cmap 
    )

    # Add reference lines and formatting
    ax.axhline(y=depth_bins[-1], color="k", linewidth=1)
    ax.set_xlim(time_segment[0], time_segment[-1])
    ax.set_ylim(np.max(depth[cluster_indices]), depth_bins[0])
    ax.axvline(x=1, color="w", linestyle="-", linewidth=1)  # Vertical line at time 1
    ax.axvline(x=2, color="w", linestyle="-", linewidth=1)  # Vertical line at time 2

    # Add cortical layer strip
    layer_colors = {
        "supragranular": '#FF8000', 
        "granular": '#008080', 
        "infragranular": '#4B0082'
    }
    unique_layers = np.unique(binned_layers)
    layer_strip = np.array([list(layer_colors.keys()).index(l) for l in binned_layers])
    layer_ax = ax.inset_axes([1.05, 0, 0.06, 1], transform=ax.transAxes)
    layer_ax.imshow(
        layer_strip[:, np.newaxis], 
        aspect='auto', 
        extent=[0, 1, depth_bins[-1], depth_bins[0]], 
        cmap=matplotlib.colors.ListedColormap(list(layer_colors.values()))
    )
    layer_ax.set_xticks([])
    layer_ax.set_yticks([])

    # Plot horizontal lines to divide different layers
    for i in range(1, len(binned_layers)):
        if binned_layers[i] != binned_layers[i - 1]:
            ax.axhline(y=depth_bins[i], color='w', linewidth=1.5)

    # Add text labels for each layer
    for layer_name, color in layer_colors.items():
        layer_midpoint = np.mean(binned_depth[binned_layers == layer_name])
        ax.text(3.4, layer_midpoint, layer_name.strip("'"), 
                va='bottom', ha='left', color=color, fontsize=8, rotation=0)

print("Plotting functions defined successfully!")

Plotting functions defined successfully!


## 4. Load and Preprocess Neural Data

In [4]:
# Load data from pickle file
# Load data from pickle file
Data = loadmat('../processed_data/Neuronal_data_10ms.mat')

Cl = {
    "Neural_Data_normalized_Ordered": Data["Neurons_activity_condition_area"],
    "Areas_Ordered": Data["Area"],
    "Type_Ordered": Data["Type"],
    "Depth_Ordered": Data["Depth"],
    "Layer_Ordered": Data["Layer"],
    "CCF_location_Ordered": Data["CCF_location"],
    "CCF_xyz_Ordered": Data["CCF_xyz"],
}
# Extract raw structures
bin_sz = 0.01  # 10 ms bin size
params_gaussiansmoothing = 0  # Keep raw firing rates (no smoothing)
baseline_subtraction=1
psth = Cl["Neural_Data_normalized_Ordered"] / bin_sz
depth = np.asarray(Cl["Depth_Ordered"], dtype=float)
ccf_location = Cl["CCF_location_Ordered"]
area = Cl["Areas_Ordered"]
Layer = Cl["Layer_Ordered"]

# Process layer information
layer = np.array([str(l.item()) if isinstance(l, np.ndarray) else str(l) 
                 for l in Layer], dtype=str)
                 
time2plot = np.arange(1, int(15 * (1 / bin_sz))) / (1 / bin_sz)

# Replace specific layers with standardized names
layer = np.array([
    l.strip("[]").replace('6a', 'infragranular').replace('6b', 'infragranular')
    .replace('5', 'infragranular').replace('4', 'granular')
    .replace('1', 'supragranular').replace('2/3', 'supragranular') 
    for l in layer
], dtype=str)
def _to_clean_string(value):
    if isinstance(value, (list, tuple)) and value:
        value = value[0]
    if isinstance(value, np.ndarray):
        if value.shape == ():
            value = value.item()
        elif value.size > 0:
            value = value.flat[0]
    text = str(value)
    text = text.replace("[", "").replace("]", "").replace("'", "").strip()
    return text

layer = np.array([_to_clean_string(l) for l in layer], dtype=str)
# Get unique layers after replacement
unique_layers = np.unique(layer)
print(f"Unique layers found: {unique_layers}")

# Set color limits for plotting
cmin, cmax = -5, 5

# Calculate z-score normalization
if baseline_subtraction:
    print("Calculating z-scores...")
    baseline_periods = np.concatenate((
        psth[:, :99], psth[:, 300:399], psth[:, 600:699], 
        psth[:, 900:999], psth[:, 1200:1299]
    ), axis=1)

    mean_psth = np.mean(baseline_periods, axis=1, keepdims=True)
    std_psth = np.std(baseline_periods, axis=1, keepdims=True)

    # Replace zero standard deviation with epsilon to avoid division by zero
    epsilon = 1e-10
    std_psth[std_psth == 0] = epsilon
    psth = (psth - mean_psth)

# Extract additional data
celltype = Cl["Type_Ordered"]
depth = Cl["Depth_Ordered"]
ccf_location = Cl["CCF_location_Ordered"]
bin_sz = 0.01  # 10 ms bin size

# Apply Gaussian smoothing if enabled
if params_gaussiansmoothing == 1:
    print("Applying Gaussian smoothing...")
    gauss_window = 0.02  # 20 ms window
    N = int(gauss_window / bin_sz)
    if N > 1:
        g = gaussian(N, std=(N / 6.0))
        w = g / np.trapz(g)  # Normalize area = 1
        psth = np.array([convolve(row, w, mode='same') for row in psth])

print(f"Data preprocessing completed. PSTH shape: {psth.shape}")

Unique layers found: ['granular' 'infragranular' 'supragranular']
Calculating z-scores...
Data preprocessing completed. PSTH shape: (10294, 1500)


In [5]:
# Statistical Analysis of Layer-Specific Activity

# Define time window for analysis (0.8 to 1 second)
time_window_start = 1.8  # seconds
time_window_end =2   # seconds

# Convert time window to bin indices
start_bin = int(time_window_start / bin_sz)-1
end_bin = int(time_window_end / bin_sz)-1

print(f"Analyzing time window: {time_window_start}s to {time_window_end}s")
print(f"Bin range: {start_bin} to {end_bin}")

# Initialize storage for results
layer_activity_results = {}
statistical_results = {}

# Define brain areas and layers
brain_areas = ['A1', 'wS1', 'wS2', 'wM2', 'ALM']
layer_types = ["supragranular", "granular", "infragranular"]

# Process each brain area
for area_name in brain_areas:
    print(f"\n--- Analyzing {area_name} ---")
    
    # Get indices for current area
    area_indices = np.where(area == area_name)[0]
    
    if len(area_indices) == 0:
        print(f"No neurons found in {area_name}")
        continue
    
    # Get layer information for this area
    area_layers = layer[area_indices]
    
    # Store activity for each layer
    layer_activities = {}
    
    # Process each layer
    for layer_type in layer_types:
        # Find neurons in this area and layer
        layer_indices = area_indices[np.where(area_layers == layer_type)[0]]
        
        if len(layer_indices) == 0:
            print(f"  {layer_type}: No neurons found")
            layer_activities[layer_type] = np.array([])
            continue
        
        # Get PSTH data for neurons in this layer
        layer_psth = psth[layer_indices, start_bin:end_bin]
        
        # Calculate mean activity for each neuron in the time window
        neuron_activities = np.nanmean(layer_psth, axis=1)
        
        # Remove any NaN values
        neuron_activities = neuron_activities[~np.isnan(neuron_activities)]
        
        layer_activities[layer_type] = neuron_activities
        
        if len(neuron_activities) > 0:
            mean_activity = np.nanmean(neuron_activities)
            std_activity = np.nanstd(neuron_activities)
        else:
            mean_activity = np.nan
            std_activity = np.nan
        print(f"  {layer_type}: {len(neuron_activities)} neurons, mean activity = {mean_activity:.3f} ± {std_activity:.3f}")
    
    # Store results for this area
    layer_activity_results[area_name] = layer_activities
    
    # Perform statistical tests between layers
    from scipy import stats
    
    area_stats = {}
    
    # Compare all pairs of layers
    layer_pairs = [
        ("supragranular", "granular"),
        ("supragranular", "infragranular"),
        ("granular", "infragranular")
    ]
    
    for layer1, layer2 in layer_pairs:
        data1 = layer_activities[layer1]
        data2 = layer_activities[layer2]
        
        if len(data1) > 0 and len(data2) > 0:
            # Perform t-test
            t_stat, p_value = stats.ttest_ind(data1, data2)
            
            # Calculate effect size (Cohen's d)
            pooled_std = np.sqrt(((len(data1) - 1) * np.var(data1) + (len(data2) - 1) * np.var(data2)) / 
                                (len(data1) + len(data2) - 2))
            cohens_d = (np.mean(data1) - np.mean(data2)) / pooled_std if pooled_std > 0 else 0
            
            area_stats[f"{layer1}_vs_{layer2}"] = {
                't_stat': t_stat,
                'p_value': p_value,
                'cohens_d': cohens_d,
                'mean1': np.mean(data1),
                'mean2': np.mean(data2),
                'n1': len(data1),
                'n2': len(data2)
            }
            
            significance = ""
            if p_value < 0.001:
                significance = "***"
            elif p_value < 0.01:
                significance = "**"
            elif p_value < 0.05:
                significance = "*"
            
            print(f"    {layer1} vs {layer2}: t={t_stat:.3f}, p={p_value:.4f}{significance}, d={cohens_d:.3f}")
    
    statistical_results[area_name] = area_stats

print("\n" + "="*80)
print("STATISTICAL SUMMARY TABLE")
print("="*80)
print("Layer-specific activity analysis (time window: 0.8-1.0s)")
print("Format: Mean ± SEM (n neurons)")
print()

# Create summary table
import pandas as pd

# Prepare data for table
table_data = []

for area_name in brain_areas:
    if area_name in layer_activity_results:
        area_data = layer_activity_results[area_name]
        row = {'Region': area_name}
        
        for layer_type in layer_types:
            activities = area_data[layer_type]
            display_key = layer_type.capitalize()
            if len(activities) > 0:
                mean_activity = np.nanmean(activities)
                sem_activity = stats.sem(activities, nan_policy='omit')
                row[display_key] = f"{mean_activity:.3f} ± {sem_activity:.3f} (n={len(activities)})"
            else:
                row[display_key] = "No data"
        
        table_data.append(row)

# Create and display DataFrame
df = pd.DataFrame(table_data)
print(df.to_string(index=False))

print("\n" + "="*80)
print("PAIRWISE COMPARISONS (t-tests)")
print("="*80)
print("Format: t-statistic, p-value, Cohen's d, significance")
print("Significance: *** p<0.001, ** p<0.01, * p<0.05")
print()

# Create comparison table
comparison_data = []

for area_name in brain_areas:
    if area_name in statistical_results:
        area_stats = statistical_results[area_name]
        
        for comparison, stats_dict in area_stats.items():
            layer1, layer2 = comparison.split('_vs_')
            significance = ""
            p_val = stats_dict['p_value']
            
            if p_val < 0.001:
                significance = "***"
            elif p_val < 0.01:
                significance = "**"
            elif p_val < 0.05:
                significance = "*"
            
            # Clean layer names for display
            layer1_clean = layer1.capitalize()
            layer2_clean = layer2.capitalize()
            
            comparison_data.append({
                'Region': area_name,
                'Comparison': f"{layer1_clean} vs {layer2_clean}",
                't-statistic': f"{stats_dict['t_stat']:.3f}",
                'p-value': f"{stats_dict['p_value']:.4f}",
                'Cohen\'s d': f"{stats_dict['cohens_d']:.3f}",
                'Significance': significance,
                'n1': stats_dict['n1'],
                'n2': stats_dict['n2']
            })

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

print("\n" + "="*80)
print("SIGNIFICANT RESULTS SUMMARY")
print("="*80)
print("Detailed statistics for statistically significant comparisons:")
print()

# Find and display significant results
significant_comparisons = []

for area_name in brain_areas:
    if area_name in statistical_results:
        area_stats = statistical_results[area_name]
        area_data = layer_activity_results[area_name]
        
        for comparison, stats_dict in area_stats.items():
            if stats_dict['p_value'] < 0.05:  # Significant results
                layer1, layer2 = comparison.split('_vs_')
                layer1_clean = layer1.capitalize()
                layer2_clean = layer2.capitalize()
                
                # Get detailed statistics
                data1 = area_data[layer1]
                data2 = area_data[layer2]
                
                mean1 = np.nanmean(data1)
                std1 = np.nanstd(data1)
                sem1 = stats.sem(data1, nan_policy='omit')
                n1 = len(data1)
                
                mean2 = np.nanmean(data2)
                std2 = np.nanstd(data2)
                sem2 = stats.sem(data2, nan_policy='omit')
                n2 = len(data2)
                
                # Determine significance level
                significance = ""
                if stats_dict['p_value'] < 0.001:
                    significance = "***"
                elif stats_dict['p_value'] < 0.01:
                    significance = "**"
                elif stats_dict['p_value'] < 0.05:
                    significance = "*"
                
                print(f"🔍 {area_name}: {layer1_clean} vs {layer2_clean} {significance}")
                print(f"   p-value = {stats_dict['p_value']:.10f}")
                print(f"   t-statistic = {stats_dict['t_stat']:.3f}")
                print(f"   Cohen's d = {stats_dict['cohens_d']:.3f}")
                print(f"   ")
                print(f"   {layer1_clean}: Mean = {mean1:.3f} ± {sem1:.3f} (SEM), SD = {std1:.3f}, n = {n1}")
                print(f"   {layer2_clean}: Mean = {mean2:.3f} ± {sem2:.3f} (SEM), SD = {std2:.3f}, n = {n2}")
                print(f"   Difference: {mean1:.3f} - {mean2:.3f} = {mean1-mean2:.3f}")
                print()
                
                significant_comparisons.append({
                    'region': area_name,
                    'comparison': f"{layer1_clean} vs {layer2_clean}",
                    'p_value': stats_dict['p_value'],
                    'significance': significance
                })

if not significant_comparisons:
    print("No statistically significant differences found between cortical layers.")
else:
    print(f"Found {len(significant_comparisons)} statistically significant comparison(s).")

print("\n" + "="*80)
print("ANALYSIS COMPLETE")
print("="*80)


Analyzing time window: 1.8s to 2s
Bin range: 179 to 199

--- Analyzing A1 ---
  supragranular: 65 neurons, mean activity = 0.238 ± 1.159
  granular: 79 neurons, mean activity = 0.491 ± 3.254
  infragranular: 636 neurons, mean activity = 0.287 ± 2.088
    supragranular vs granular: t=-0.592, p=0.5551, d=-0.100
    supragranular vs infragranular: t=-0.186, p=0.8527, d=-0.024
    granular vs infragranular: t=0.759, p=0.4480, d=0.091

--- Analyzing wS1 ---
  supragranular: 299 neurons, mean activity = -0.104 ± 1.376
  granular: 529 neurons, mean activity = 0.100 ± 1.461
  infragranular: 1110 neurons, mean activity = 0.135 ± 2.088
    supragranular vs granular: t=-1.964, p=0.0498*, d=-0.142
    supragranular vs infragranular: t=-1.869, p=0.0618, d=-0.122
    granular vs infragranular: t=-0.347, p=0.7283, d=-0.018

--- Analyzing wS2 ---
  supragranular: 460 neurons, mean activity = -0.052 ± 1.429
  granular: 421 neurons, mean activity = 0.106 ± 1.531
  infragranular: 1183 neurons, mean activ

## 5. Create Main Figure and Plot Heatmaps

In [6]:
# Create main figure with subplots
print("Creating figure...")
fig, axs = plt.subplots(nrows=2, ncols=5, figsize=(50 / 2.54, 15 / 2.54))

# Define brain areas to analyze
unique_areas = [['A1'], ['wS1'], ['wS2'], ['wM2'], ['ALM']]

# Plot heatmaps for each brain area (middle column)
print("Plotting depth-resolved heatmaps...")
for col, unique_area in enumerate(unique_areas):
    area_indices = np.where(area == unique_area)[0]
    start_bin = 0 * 300
    end_bin = start_bin + 300
    
    plot_heatmap(axs[0, col], psth, area_indices, time2plot, 
                start_bin, end_bin, depth, layer)
    
    axs[0, col].set_title(f"{unique_area[0]}")
    axs[0, col].set_xticks([0, 1, 2, 3])
    axs[0, col].set_xticklabels([0, 1, 2, 3])
    axs[0, col].set_xlim([0, 3])
    
    

# Add colorbar
cbar_ax = fig.add_axes([.4, .5, 0.05, 0.01])
fig.colorbar(
    axs[0, 0].images[0],
    cax=cbar_ax,
    orientation='horizontal',
)

print("Heatmaps completed!")

Creating figure...
Plotting depth-resolved heatmaps...
Heatmaps completed!


c:\ProgramData\anaconda3\Lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\ProgramData\anaconda3\Lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\ProgramData\anaconda3\Lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\ProgramData\anaconda3\Lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


## 6. Load Clustering Data and Process Layer Information

In [7]:
# Load clustering data for additional analysis
print("Loading clustering data...")
# Data files are located one folder up in the results directory



unique_areas = ['A1', 'wS1', 'wS2', 'wM2', 'ALM']
n_cols = len(unique_areas)
n_rows = 1

# Process layer information for clustering data
layer = np.array([str(l.item()) if isinstance(l, np.ndarray) else str(l) 
                 for l in Layer], dtype=str)
layer = np.array([
    l.strip("[]").replace('6a', 'infragranular').replace('6b', 'infragranular')
    .replace('5', 'infragranular')
    .replace('1', 'supragranular').replace('2/3', 'supragranular').replace('4', 'granular') 
    for l in layer
], dtype=str)

# Define layer colors
layer_colors = {
    "'supragranular'": '#B87333', 
    "'granular'": '#008080', 
    "'infragranular'": '#4B0082'
}

print("Clustering data loaded and processed!")

Loading clustering data...
Clustering data loaded and processed!


## 7. Plot Layer-Specific PSTHs (Right Column)

In [8]:
# Plot layer-specific PSTHs (right column)
print("Plotting layer-specific PSTHs...")
for ax, unique_area in zip(axs[1,:], unique_areas):
    area_indices = np.where(area == unique_area)[0]
    unique_layers = np.unique(layer[area_indices])

    for unique_layer in unique_layers:
        layer_indices = area_indices[np.where(layer[area_indices] == unique_layer)[0]]
        rows_for_layer = psth[layer_indices, :]

        if rows_for_layer.size == 0:
            continue

        # Calculate mean and standard error
        mean_sig = np.nanmean(rows_for_layer, axis=0)
        sem_sig = np.nanstd(rows_for_layer, axis=0) / np.sqrt(rows_for_layer.shape[0])
        curve1, curve2 = mean_sig + sem_sig, mean_sig - sem_sig

        seg_len = int(3 * (1 / bin_sz))  # Length of each segment (300 bins)
        x_segment = time2plot[:seg_len]  # Use same x range for all conditions

        for i_cond in range(1):  # Iterate over conditions
            st, en = i_cond * seg_len, (i_cond + 1) * seg_len
            y_mean = mean_sig[st:en]
            y_upper = curve1[st:en]
            y_lower = curve2[st:en]

            # Plot with error shading
            ax.fill_between(x_segment, y_lower, y_upper, alpha=0.2, 
                          linewidth=0, color=layer_colors[unique_layer])
            ax.plot(x_segment, y_mean, linewidth=1, 
                   label=f"{unique_layer}", color=layer_colors[unique_layer])

    # Add formatting
    ax.axvline(x=1, color='k', linestyle='-', linewidth=1)
    ax.axvline(x=2, color='k', linestyle='-', linewidth=1)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_title(f" {unique_area}", fontsize=10)
    ax.set_xticks([0, 1, 2, 3])
    ax.set_xticklabels([0, 1, 2, 3])
    ax.set_xlim([0, 3])

# Add legend and set y-axis limits
axs[1, 0].legend(loc='upper right', fontsize=6)
axs[1, 4].set_ylim([-2, 6])
axs[1, 3].set_ylim([-2, 6])
axs[1, 2].set_ylim([-2, 15])
axs[1, 1].set_ylim([-2, 15])
axs[1, 0].set_ylim([-2, 15])

print("Layer-specific PSTHs completed!")

Plotting layer-specific PSTHs...
Layer-specific PSTHs completed!


## 9. Final Formatting and Export

In [9]:
# Final formatting and export
print("Finalizing figure...")
plt.tight_layout()
plt.show(block=False)

# Save figure
output_filename = "../Main_figures_pdf/Ghaderi2025_Figure3_d_PSTH_depth.svg"
fig.savefig(output_filename, format='svg', dpi=300, bbox_inches='tight')
print(f"Figure saved as: {output_filename}")
print("\nAnalysis completed successfully!")

Finalizing figure...


C:\Users\crochet\AppData\Local\Temp\ipykernel_37436\2911537988.py:3: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
posx and posy should be finite values
posx and posy should be finite values
posx and posy should be finite values
posx and posy should be finite values


Figure saved as: ../Main_figures_pdf/Ghaderi2025_Figure3_d_PSTH_depth.svg

Analysis completed successfully!


posx and posy should be finite values
posx and posy should be finite values
